# Почему baseline обязателен до любой «серьезной» модели

Когда инженер видит табличный набор данных и хочет построить прогноз, естественным желанием становится сразу перейти к более сложному алгоритму. Однако без baseline и корректной схемы разбиения качество модели оказывается числом без контекста.

<div style="border-left: 4px solid #1f4b99; background: #eef5ff; padding: 12px 14px; margin: 12px 0;">
<strong>Учебная задача.</strong><br/>
Нужно показать на тематическом энергетическом наборе, как организуется честная схема оценки: данные разделяются на обучающую и тестовую части, baseline задает точку отсчета, а содержательная модель сравнивается с ним по нескольким метрикам.
</div>

## Исходные данные

Используется набор <em>Combined Cycle Power Plant</em>, в котором по атмосферным параметрам требуется оценивать электрическую мощность на выходе станции. Это удобный пример для первой регрессионной постановки: признаки понятны физически, а задача имеет прямую инженерную интерпретацию.

## Ожидаемый результат

После выполнения блокнота должно быть понятно:

- как устроено разбиение на обучающую и тестовую выборки;
- зачем нужен baseline-прогноз;
- как интерпретировать MAE, RMSE и $R^2$;
- почему даже хороший численный результат нужно соотносить с практическим смыслом ошибки.

## Логика эксперимента

### 1. Загрузка локального тематического набора
Файл открывается из `data/sample/regression/`.

### 2. Подготовка признаков и целевой переменной
Определяются входные параметры и прогнозируемая величина.

### 3. Разбиение и сравнение двух уровней модели
Сначала baseline, затем линейная регрессия.

### 4. Интерпретация метрик
Полученные числа рассматриваются не сами по себе, а как оценка инженерной пригодности прогноза.

In [ ]:
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from src.display_utils import display_callout, display_metric_table
from src.project_paths import sample_data_dir

DATA_PATH = sample_data_dir() / "regression" / "combined_cycle_power_plant.csv"
df = pd.read_csv(DATA_PATH)
df.head()

## Первое чтение таблицы

Уже по первым строкам видно, что набор состоит из четырех входных признаков и одной целевой величины. Такая структура делает его удобным для демонстрации базовой регрессионной постановки и сравнения моделей.

In [ ]:
summary = pd.DataFrame(
    {
        "rows": [len(df)],
        "features": [4],
        "target": ["PE"],
        "missing_values": [int(df.isna().sum().sum())],
    }
)
summary

## Почему этот набор удобен методически

Набор содержит достаточно наблюдений для учебного эксперимента, не перегружен лишними полями и не требует сложной очистки перед первым сравнением моделей. Это позволяет сосредоточиться именно на логике оценки качества.

In [ ]:
X = df[["AT", "V", "AP", "RH"]]
y = df["PE"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

split_table = pd.DataFrame(
    {
        "subset": ["train", "test"],
        "rows": [len(X_train), len(X_test)],
    }
)
split_table

## Интерпретация разбиения

Тестовая выборка отделяется заранее, чтобы итоговая оценка не смешивалась с этапом подбора модели. Это ключевой элемент любого воспроизводимого вычислительного эксперимента.

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
model_pred = linear_model.predict(X_test)

comparison = pd.DataFrame(
    {
        "model": ["baseline_mean", "linear_regression"],
        "MAE": [
            mean_absolute_error(y_test, base_pred),
            mean_absolute_error(y_test, model_pred),
        ],
        "RMSE": [
            mean_squared_error(y_test, base_pred) ** 0.5,
            mean_squared_error(y_test, model_pred) ** 0.5,
        ],
        "R2": [
            r2_score(y_test, base_pred),
            r2_score(y_test, model_pred),
        ],
    }
)
comparison

## Что показывает сравнение моделей

Baseline задает минимальный ориентир: если содержательная модель не улучшает его, то ее усложнение не имеет смысла. В данном случае линейная регрессия дает существенно более качественный результат, что делает сравнение содержательным и методически корректным.

In [ ]:
display_metric_table(
    {
        "baseline_mae": float(comparison.loc[0, "MAE"]),
        "linear_mae": float(comparison.loc[1, "MAE"]),
        "baseline_rmse": float(comparison.loc[0, "RMSE"]),
        "linear_rmse": float(comparison.loc[1, "RMSE"]),
    },
    decimals=3,
)

display_callout(
    "Инженерный смысл метрик",
    "Полученные ошибки следует соотносить с допустимым диапазоном отклонения мощности, а не рассматривать только как абстрактные числа качества модели.",
    tone="success",
)

## Итоговый вывод

Схема «разбиение -> baseline -> содержательная модель -> интерпретация метрик» является обязательной основой для последующих кейсов. Без нее сравнение моделей теряет методическую строгость и инженерный смысл.

## Вопросы для самостоятельной работы

1. Почему baseline со средним значением обязателен даже в простом учебном примере?
2. В каких ситуациях MAE более удобна для инженерной интерпретации, чем RMSE?
3. Как изменилась бы схема оценки, если бы данные представляли временной ряд?

## Источники

1. [Глава 4 пособия](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/chapters/04-splitting-metrics-and-baselines.md)
2. [Паспорт датасетов](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/datasets.md)